In [49]:
# Importing necessary libraries
import pandas as pd
import sqlite3

## Task 1: Data Gathering and Combination

In [50]:
# Establishing a connection to the SQLite database
conn = sqlite3.connect('level3_final_project_database.db')

In [51]:
# Exploring the members table
members_df = pd.read_sql_query("SELECT * FROM members", conn)
print("Members Table:")
print(members_df.head())

Members Table:
   member_id first_name last_name  grade neighborhood membership_status  \
0       1001      Salma   Ibrahim    8.0        Maadi            Active   
1       1002      Fares     Saleh    9.0        Maadi            Active   
2       1003     Bassel    Hegazy    6.0        Maadi            Active   
3       1004      Fares     Wahba    7.0        Maadi          inactive   
4       1005    Youssef     Halim    9.0        Maadi            Active   

    join_date  
0  2023-04-05  
1         NaN  
2  2025-04-23  
3  2024-10-09  
4  2024-05-05  


In [52]:
# Exploring checkouts table
checkouts_df = pd.read_sql_query("SELECT * FROM checkouts", conn)
print("\nCheckouts Table:")
print(checkouts_df.head())


Checkouts Table:
   checkout_id  member_id  book_id checkout_date return_date
0         9263       1047      517    2024-10-21  2024-11-07
1         9340       1072      513    2025-08-24  2025-09-01
2         9231       1053      523    2024-02-04  2024-02-16
3         9129       1032      513    2025-06-21  2025-06-29
4         9370       1079      511    2025-11-11  2025-12-03


In [53]:
# Exploring books table
books_df = pd.read_sql_query("SELECT * FROM books", conn)
print("\nBooks Table:")
print(books_df.head())


Books Table:
   book_id                title         author
0      501      The Silver Kite  Amina Darwish
1      502       Desert Compass  Amina Darwish
2      503    The Lantern Maker   Adel Roushdy
3      504  Rooftop Astronomers   Adel Roushdy
4      505  Letters to the Nile      Aya Hafez


In [54]:
# First question: How much is each member borrowing?
each_borrowing_count = pd.read_sql_query('''
SELECT
    members.member_id,
    members.first_name,
    members.last_name,
    COUNT(checkouts.checkout_id) AS borrowing_count
FROM members
LEFT JOIN checkouts ON members.member_id = checkouts.member_id
GROUP BY members.member_id, members.first_name, members.last_name
''', conn)
print("\nMember Checkouts Table:")
print(each_borrowing_count.head())


Member Checkouts Table:
   member_id first_name last_name  borrowing_count
0       1001      Salma   Ibrahim                1
1       1002      Fares     Saleh                2
2       1003     Bassel    Hegazy                9
3       1004      Fares     Wahba                0
4       1005    Youssef     Halim                3


In [55]:
# Second question: Which books match a chosen author pattern?
pattern_authors_df = pd.read_sql_query("SELECT * FROM books WHERE author LIKE 'A%'", conn)
print("\nBooks by Authors Starting with 'A':")
print(pattern_authors_df)


Books by Authors Starting with 'A':
   book_id                title         author
0      501      The Silver Kite  Amina Darwish
1      502       Desert Compass  Amina Darwish
2      503    The Lantern Maker   Adel Roushdy
3      504  Rooftop Astronomers   Adel Roushdy
4      505  Letters to the Nile      Aya Hafez
5      506  The Paper Boat Club      Aya Hafez


In [56]:
# Third question: What are the most five popular books?
popular_books_df = pd.read_sql_query('''
SELECT
    books.title,
    COUNT(checkouts.checkout_id) as checkout_count
FROM books
LEFT JOIN checkouts ON books.book_id = checkouts.book_id
GROUP BY books.title, books.book_id 
ORDER BY checkout_count DESC
LIMIT 5''', conn)
print("\nTop 5 Most Popular Books:")
print(popular_books_df)


Top 5 Most Popular Books:
                    title  checkout_count
0         The Silver Kite              57
1   Fossils and Fireflies              55
2  Circuits for Beginners              46
3        Kites Over Cairo              38
4    Storms and Sailboats              25


In [57]:
# Fourth question: Who are the most active readers?
active_readers_df = pd.read_sql_query('''
SELECT 
    members.first_name,
    members.last_name,
    members.member_id,
    COUNT(checkouts.checkout_id) as Borrowing_count
FROM members
JOIN checkouts ON members.member_id = checkouts.member_id
GROUP BY members.member_id ORDER BY Borrowing_count DESC 
LIMIT 10;
''', conn)
print("\nTop 10 Most Active Readers:")
print(active_readers_df)


Top 10 Most Active Readers:
  first_name last_name  member_id  Borrowing_count
0        Aya     Wahba       1034               25
1     Sherif     Saleh       1044               21
2       Ziad     Saleh       1008               19
3    Mostafa     Fouad       1027               18
4       Nour     Nabil       1010               18
5       Adam     Fahmy       1065               17
6    Youssef    Hegazy       1024               17
7      Ahmed    Shafik       1018               17
8       Sara    Rashad       1047               16
9       Reem     Osman       1030               16


In [58]:
# Fifth question: What does a neighborhood's activity look like further back in time?
neighborhood_activity_df = pd.read_sql_query("""
SELECT 
    checkouts.checkout_date,
    members.first_name,
    members.last_name,
    members.member_id
FROM checkouts
JOIN members ON checkouts.member_id = members.member_id
WHERE members.neighborhood = 'Maadi'
ORDER BY checkouts.checkout_date DESC
LIMIT 10
OFFSET 10;
""", conn)
print("\nNeighborhood Activity for 'Maadi':")
print(neighborhood_activity_df)


Neighborhood Activity for 'Maadi':
  checkout_date first_name last_name  member_id
0    2025-09-04     Bassel    Hegazy       1003
1    2025-08-25       Adam      Badr       1017
2    2025-08-23       Ziad     Saleh       1008
3    2025-08-21     Bassel    Hegazy       1003
4    2025-08-19      Ahmed    Shafik       1018
5    2025-08-08      Hamza     Sabry       1015
6    2025-08-04      Ahmed    Shafik       1018
7    2025-07-27       Ziad     Fouad       1013
8    2025-07-22     Bassel    Hegazy       1003
9    2025-07-22     Hassan     Saleh       1009


In [59]:
# Closing the connection with the database
conn.close()

In [60]:
# Stage1: Merging members and checkouts table
stage1 = pd.merge(members_df, checkouts_df, on='member_id', how='left')
print("Stage1 DataFrame\n")
print(stage1.head())

Stage1 DataFrame

   member_id first_name last_name  grade neighborhood membership_status  \
0       1001      Salma   Ibrahim    8.0        Maadi            Active   
1       1002      Fares     Saleh    9.0        Maadi            Active   
2       1002      Fares     Saleh    9.0        Maadi            Active   
3       1003     Bassel    Hegazy    6.0        Maadi            Active   
4       1003     Bassel    Hegazy    6.0        Maadi            Active   

    join_date  checkout_id  book_id checkout_date return_date  
0  2023-04-05       9025.0    525.0    2024-10-24  2024-11-07  
1         NaN       9013.0    501.0    2024-02-16  2024-02-29  
2         NaN       9095.0    507.0    2024-06-25  2024-07-07  
3  2025-04-23       9051.0    513.0    2025-07-22  2025-08-18  
4  2025-04-23       9068.0    501.0    2025-11-14  2025-12-03  


In [61]:
# Book Details
book_details = pd.read_json('level3_final_project_book_catalog')
print("Book details:\n")
print(book_details.head())

Book details:

   book_id       genre  pages  publication_year            publisher
0      501   Adventure    128            2017.0           Nile Press
1      502   Adventure    109            2018.0          Delta House
2      503  Historical    259               NaN           Nile Press
3      504     Science    319            2009.0  Cairo Young Readers
4      505  Historical    216            2024.0          Oasis Books


In [62]:
# Stage2: Merge The Two DataFrames 
stage2 = pd.merge(stage1, book_details, on='book_id', how='left')
print("Stage2 DataFrame:\n")
print(stage2.head())

Stage2 DataFrame:

   member_id first_name last_name  grade neighborhood membership_status  \
0       1001      Salma   Ibrahim    8.0        Maadi            Active   
1       1002      Fares     Saleh    9.0        Maadi            Active   
2       1002      Fares     Saleh    9.0        Maadi            Active   
3       1003     Bassel    Hegazy    6.0        Maadi            Active   
4       1003     Bassel    Hegazy    6.0        Maadi            Active   

    join_date  checkout_id  book_id checkout_date return_date      genre  \
0  2023-04-05       9025.0    525.0    2024-10-24  2024-11-07  Adventure   
1         NaN       9013.0    501.0    2024-02-16  2024-02-29  Adventure   
2         NaN       9095.0    507.0    2024-06-25  2024-07-07    Science   
3  2025-04-23       9051.0    513.0    2025-07-22  2025-08-18    Science   
4  2025-04-23       9068.0    501.0    2025-11-14  2025-12-03  Adventure   

   pages  publication_year    publisher  
0  297.0            2015.0  Oas

In [63]:
# Scraping the HTML page to get event signup
scraped_df = pd.read_html('level3_final_project_event_signup')[0]
print("Scraped Event Signup DataFrame:\n")
print(scraped_df)

Scraped Event Signup DataFrame:

    Member ID  Book ID Checkout Date
0        1026      522    2025-07-11
1        1049      520    2025-07-11
2        1062      525    2025-07-05
3        1065      520    2025-07-07
4        1104      515    2025-07-07
5        1009      503    2025-07-09
6        1063      522    2025-07-07
7        1022      511    2025-07-12
8        1029      523    2025-07-09
9        1201      509    2025-07-10
10       1005      513    2025-07-10
11       1104      526    2025-07-05
12       1058      518    2025-07-08
13       1002      521    2025-07-08
14       1150      530    2025-07-10
15       1041      504    2025-07-05
16       1055      526    2025-07-11
17       1061      513    2025-07-06
18       1012      509    2025-07-11
19       1007      510    2025-07-07
20       1073      530    2025-07-07
21       1003      501    2025-07-08
22       1017      507    2025-07-11
23       1061      504    2025-07-06
24       1201      523    2025-07-08
25   

In [64]:
# Adjust the name of the columns of the scraped dataframe
print("Old Column Names:")
print(scraped_df.columns.tolist())

scraped_df.columns = [col.strip().lower().replace(' ', '_') for col in scraped_df.columns]

print("New Column Names:")
print(scraped_df.columns.tolist())


Old Column Names:
['Member ID', 'Book ID', 'Checkout Date']
New Column Names:
['member_id', 'book_id', 'checkout_date']


In [65]:
# Merge the scraped df to be ready to concat with the stage2 df
scraped_full = pd.merge(scraped_df, members_df, on='member_id', how='left')
scraped_full = pd.merge(scraped_full, book_details, on='book_id', how='left')

print(scraped_full.head())

   member_id  book_id checkout_date first_name last_name  grade neighborhood  \
0       1026      522    2025-07-11       Nada     Saleh    7.0    Nasr City   
1       1049      520    2025-07-11      Ahmed     Gamal    6.0   Heliopolis   
2       1062      525    2025-07-05      Tarek      Adel    7.0      Zamalek   
3       1065      520    2025-07-07       Adam     Fahmy    6.0      Zamalek   
4       1104      515    2025-07-07        NaN       NaN    NaN          NaN   

  membership_status   join_date            genre  pages  publication_year  \
0          inactive  2023-11-16          Mystery    104            2016.0   
1            Active  2023-06-10  Science Fiction    136            2009.0   
2            Active  2023-01-22        Adventure    297            2015.0   
3            Active  2025-07-12  Science Fiction    136            2009.0   
4               NaN         NaN           Poetry    316            2022.0   

             publisher  
0  Cairo Young Readers  
1     

In [66]:
# Concat the stage2 df and the scraped_full df

stage3 = pd.concat([stage2, scraped_full], ignore_index=True)
print("Stage3 DataFrame:\n")
print(stage3.head())
print('--' * 50)
print('The shape of the DataFrame(rows ,columns):',stage3.shape)
print('--' * 50)
print("Columns:\n",stage3.columns.tolist())

Stage3 DataFrame:

   member_id first_name last_name  grade neighborhood membership_status  \
0       1001      Salma   Ibrahim    8.0        Maadi            Active   
1       1002      Fares     Saleh    9.0        Maadi            Active   
2       1002      Fares     Saleh    9.0        Maadi            Active   
3       1003     Bassel    Hegazy    6.0        Maadi            Active   
4       1003     Bassel    Hegazy    6.0        Maadi            Active   

    join_date  checkout_id  book_id checkout_date return_date      genre  \
0  2023-04-05       9025.0    525.0    2024-10-24  2024-11-07  Adventure   
1         NaN       9013.0    501.0    2024-02-16  2024-02-29  Adventure   
2         NaN       9095.0    507.0    2024-06-25  2024-07-07    Science   
3  2025-04-23       9051.0    513.0    2025-07-22  2025-08-18    Science   
4  2025-04-23       9068.0    501.0    2025-11-14  2025-12-03  Adventure   

   pages  publication_year    publisher  
0  297.0            2015.0  Oas

In [67]:
# Exploring last five rows
print("Last five rows:\n")
print(stage3.tail())

Last five rows:

     member_id first_name last_name  grade neighborhood membership_status  \
430       1003     Bassel    Hegazy    6.0        Maadi            Active   
431       1017       Adam      Badr    9.0        Maadi            Active   
432       1061       Ziad     Fahmy    9.0      zamalek          Inactive   
433       1201        NaN       NaN    NaN          NaN               NaN   
434       1041       Nada    Rashad    7.0    Nasr City            Active   

      join_date  checkout_id  book_id checkout_date return_date       genre  \
430  2025-04-23          NaN    501.0    2025-07-08         NaN   Adventure   
431  2025-07-12          NaN    507.0    2025-07-11         NaN     Science   
432  2023-04-27          NaN    504.0    2025-07-06         NaN     Science   
433         NaN          NaN    523.0    2025-07-08         NaN  Historical   
434  2024-04-02          NaN    512.0    2025-07-06         NaN   Adventure   

     pages  publication_year            publi

In [68]:
stage3.to_csv('task1_combined_data.csv', index=False)
print("Data Saved Successfully.")

Data Saved Successfully.
